# Coding Agent Eval

In this notebook, you examine two coding datapoints from [strands-agents/evals](https://github.com/strands-agents/evals). Each datapoint is a real bug: an issue report, the commit before the fix, and the fix that the team merged. The agent gets the issue report and the repository, and it must write the fix.

A fix that passes the tests is not always a fix that a maintainer would merge. So the verifier grades the fix in the same way as [Senior SWE-Bench](https://senior-swe-bench.snorkel.ai/blog/2026-06-16-how-it-works): hidden tests, tests that a validation agent writes for the agent's change, a task rubric, a taste judge, and patch bloat. A task is solved only if the fix is correct *and* tasteful.

The datapoints use the same format as [Senior SWE-Bench](https://hub.harborframework.com/datasets/snorkel-ai/senior-swe-bench-v2026.06/latest?tab=tasks), for example the task [gitea-fix-force-push-timeline](https://hub.harborframework.com/tasks/snorkel-ai/gitea-fix-force-push-timeline). The datapoints are in `datasets/strands-evals-coding/`. We made them before the workshop.

This notebook does not use the datapoints of notebook 02. It uses the same Harbor setup: the Bedrock settings, the network allowlist, and Codex as the agent under test.

### Procedure

1. Learn what a coding datapoint is.
2. Examine the two datapoints.
3. Examine the task files and do a leakage scan.
4. Select the metrics.
5. Run the oracle, a no-op agent, and Codex in Harbor.
6. Calculate the results.
7. Look at the results in Harbor view.

This cell adds your shell `PATH` to the kernel and loads your settings from `.env`, as in notebook 02.

In [ ]:
from env_setup import fix_path, load_env; fix_path(); load_env()

In [ ]:
import json
import os
import re
import shutil
import subprocess
import tomllib
from pathlib import Path

DATASET_DIR = Path("datasets/strands-evals-coding")
TASKS_DIR = DATASET_DIR / "tasks"
JOBS_DIR = Path("jobs")
ROOT_TASK = TASKS_DIR / "strands-agents__evals__pr371-fix"
HISTORY_TASK = TASKS_DIR / "strands-agents__evals__pr338-fix"
TASK_DIRS = [ROOT_TASK, HISTORY_TASK]

# Same Bedrock settings as notebooks 01 and 02. The task.toml files allow only the Bedrock endpoint
# in us-east-1. If you change AWS_REGION, also change allowed_hosts in each task.toml.
# All model IDs of this module are in model_config.py. Harbor gives VERIFIER_MODEL_ID to the verifier
# (see [verifier.env] in each task.toml).
from model_config import CODEX_MODEL_ID, VERIFIER_MODEL_ID
os.environ["VERIFIER_MODEL_ID"] = VERIFIER_MODEL_ID
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")  # from .env, if you set it

def read_toml(path: Path) -> dict:
    return tomllib.loads(path.read_text())

def read_json(path: Path):
    return json.loads(path.read_text())

## 1. What a Coding Datapoint Is

A PR review datapoint (notebook 02) asks: *does the agent find the defects in this change?* A coding datapoint asks: *can the agent make the change?* The parts of the datapoint are:

| Part | PR review datapoint (02) | Coding datapoint (03) |
|------|--------------------------|-----------------------|
| Start state | The base commit, with the change applied | The base commit (the parent of the fix), with no change |
| Instruction | The same for all tasks: review the change | An issue report, different for each task |
| Agent output | A list of findings | Changes to the code |
| Oracle | The defects that the reviewers found | The fix that the team merged |
| Verifier | An LLM judge matches the findings to the oracle | A reward stack: tests, a validation agent, a rubric judge, and a taste judge |

The hidden tests are in two groups:

- **Fail to pass (f2p).** These tests fail on the base commit and pass with the merged fix. They show that the agent fixed the bug.
- **Pass to pass (p2p).** These tests pass on the base commit and must still pass. They show that the agent did not break something. The verifier also runs the full unit test suite of the repository at the base commit (`existing_suite`).

### The reward stack

| Mechanism | What it checks | Runs the code? | LLM? | Gate for a tasteful solve |
|-----------|----------------|----------------|------|---------------------------|
| **Pre-written verifiers** | Hidden fail-to-pass and pass-to-pass tests, and the base unit test suite | Yes | No | All pass |
| **Validation agent** | Tests that an agent writes from an expert *validation spec*, adapted to the shape of the agent's change. A validation judge reviews the tests. | Yes | Yes | All stories pass |
| **Task rubric judge** | Task-specific criteria that tests cannot check well, for example "the extractor and the trace use one rule to find the root" | No | Yes | Score > 0.5 |
| **Taste judge** | Quality compared with the merged fix (minimality, approach, hygiene, fluency, craftsmanship), and fit with the codebase (style, patterns, libraries, abstraction, documentation) | No | Yes | Each mean > 2 of 5 |
| **Patch bloat** | Source lines of the agent's change divided by source lines of the merged fix | No | No | < 2× |
| **Agent-written tests** | Did the agent add a test that fails on the base and passes on its fix? A senior engineer does this without being told. | Yes | No | Reported only |

`correct` means that the change works: the verifiers and the validation agent pass. `tasteful_solve` (the reward) also needs the rubric, taste, and bloat thresholds.

Why have both verifiers and a validation agent? A pre-written test can check only the interface that its author expected. The validation agent reads the agent's change, and writes the test for the shape that the agent built. Senior SWE-Bench uses it mainly for feature tasks, where the interface is not known before. These datapoints are bug fixes, so the verifiers already use only public interfaces. Here the validation agent tests at the system level: it runs the real evaluator with a stub model, and checks what the judge model would see.

### How the datapoints were made

1. **Select merged PRs that fix a bug.** Each PR must close an issue that describes the problem, and its tests must run without a network or a model. For example, a test that calls Amazon Bedrock cannot be a hidden test.
2. **Write the instruction from the issue.** Keep the symptom and the expected behavior. Remove the root cause, the file names, and the function names. Senior SWE-Bench writes its instructions the same way: the agent must find the cause itself.
3. **Write the hidden tests.** Do not copy the PR tests. The PR tests often import private functions, or check one design of the fix. Write tests that use only the public API, and that check only the behavior that the instruction asks for. A different correct fix must pass.
4. **Write the validation spec and the rubric.** The spec has user stories, a procedure for each story, and several cases (inputs and expected values). The rubric has fail-to-pass and pass-to-pass criteria.
5. **Check every mechanism on the oracle and on a no-op agent.** All f2p tests fail on the base, and all p2p tests pass. The oracle passes every test and every validation story, and it is a tasteful solve. The no-op agent fails. Section 5 does this in Harbor. Senior SWE-Bench runs the validation agent 3 times on each, and rejects a task unless the results agree every time (pass^3).
6. **Build a sealed environment.** Clone only the base commit, and install the dependencies. The agent cannot find the fix in the git history, and it can run the tests.

## 2. Examine the Two Datapoints

Both bugs give the same symptom: a tool-level evaluator (for example, `ToolSelectionAccuracyEvaluator`) gives false negatives. The judge says that a correct tool call is wrong, because the extractor gives it the wrong context. The causes are different.

| | Root agent datapoint | Tool history datapoint |
|---|---|---|
| Issue / PR | [#370](https://github.com/strands-agents/evals/issues/370) / [#371](https://github.com/strands-agents/evals/pull/371) | [#337](https://github.com/strands-agents/evals/issues/337) / [#338](https://github.com/strands-agents/evals/pull/338) |
| Symptom | In a flat multi-agent trace, the judge thinks that the user asked only a part of the question. | The judge says that an argument is hallucinated, but the value came from an earlier tool call. |
| Cause | The root agent is the first agent in the span list, not the agent that started first. | All tool calls in a trace get the same history, with no results of the earlier calls in that trace. |
| Size of the merged fix | 1 function, 7 lines | 2 files, approximately 40 lines |
| Hidden tests | 6 f2p, 5 p2p | 7 f2p, 6 p2p |

The datapoints have different difficulties. The fix for the root agent is small, but the agent must find which function causes the symptom. The fix for the tool history is larger, and it has edge cases: parallel tool calls, calls with equal times, and spans that are not in time order.

`[metadata]` in `task.toml` records the origin, the hidden tests, and a description of the cause and the fix. The agent does not see `task.toml`.

In [ ]:
for task_dir in TASK_DIRS:
    meta = read_toml(task_dir / "task.toml")["metadata"]
    origin, verifier, story = meta["origin"], meta["verifier"], meta["narrative"]
    print(f"{task_dir.name}")
    print(f"  origin:   {origin['repo']} PR {origin['pr_numbers']}, issue {origin['issue_numbers']}, base {origin['base_commit'][:9]}")
    print(f"  tests:    {verifier['fail_to_pass']} f2p, {verifier['pass_to_pass']} p2p, in {verifier['hidden_tests']}")
    print(f"  cause:    {' '.join(story['description'].split())}")
    print(f"  fix:      {' '.join(story['solution'].split())}")
    print()

## 3. Examine the Tasks

Each datapoint is one Harbor task directory. The agent sees `instruction.md` and the container that `environment/` builds. Harbor copies `tests/` into the container only after the agent stops. The agent never sees `solution/`.

In [ ]:
def show_tree(root: Path):
    print(root.name + "/")
    for path in sorted(root.rglob("*")):
        depth = len(path.relative_to(root).parts)
        print("    " * depth + path.name + ("/" if path.is_dir() else ""))

for task_dir in TASK_DIRS:
    show_tree(task_dir)
    print()

### `task.toml`

Make sure that:

- The agent `network_mode` is `allowlist`, and `allowed_hosts` contains only the Bedrock endpoint that Codex uses. The agent cannot reach GitHub, where the fix is, or a package index.
- The verifier runs in the agent's container (Harbor's default `shared` mode), because it must run the agent's code. This is different from notebook 02. Harbor copies `tests/` into the container only after the agent stops.
- The verifier phase has its own allowlist: only the Bedrock runtime endpoint, for the validation agent and the judges. Harbor switches the network policy when the agent stops.
- The verifier gets the AWS credentials of your shell in `[verifier.env]`. The agent phase does not get them.

In [ ]:
for task_dir in TASK_DIRS:
    cfg = read_toml(task_dir / "task.toml")
    env, verifier = cfg["environment"], cfg["verifier"]
    print(f"{task_dir.name}")
    print(f"  agent network: {env['network_mode']} {env.get('allowed_hosts', [])}")
    print(f"  verifier:      {verifier.get('environment_mode', 'shared')}, network {verifier['network_mode']} {verifier['allowed_hosts']}")
    print(f"  verifier env:  {sorted(verifier.get('env', {}))}")
    print(f"  timeouts:      agent {cfg['agent']['timeout_sec']:.0f} s, verifier {verifier['timeout_sec']:.0f} s")
    print()

    assert env["network_mode"] == "allowlist", "The agent network must be an allowlist."
    assert all("bedrock" in host for host in env["allowed_hosts"]), "Allow only the Bedrock endpoint."
    assert verifier.get("environment_mode", "shared") == "shared", "The verifier must run the agent's code."
    assert verifier["allowed_hosts"] == ["bedrock-runtime.us-east-1.amazonaws.com"], "The verifier may reach only Bedrock."

### `instruction.md`

The instruction is an issue report. It tells the symptom and the expected behavior. It does not name the file, the function, or the cause. The last part, *General instructions*, is the same for all tasks. It tells the agent where the code is, and what the container has.

In [ ]:
print((ROOT_TASK / "instruction.md").read_text())

In [ ]:
print((HISTORY_TASK / "instruction.md").read_text())

### `environment/Dockerfile`

The Dockerfile builds the container that the agent works in. Make sure that it does these steps:

1. Clones only the base commit, the parent of the fix.
2. Removes the remote and the remote refs, expires the reflog, and runs garbage collection. After this step, only one commit is in the repository.
3. Writes the base commit to `/var/lib/task_base_ref`. The verifier uses it to find the changes of the agent.
4. Installs Python and the test dependencies. The repository has no lockfile, so `requirements.lock` pins each package to the newest version on the day that the fix was merged. Without the pins, a new version of a dependency can break a test, and the datapoint changes over time.
5. **Runs the full unit test suite at build.** If a test fails on the base, the build fails. Then a failure in a trial is caused by the agent, not by the environment.
6. Installs Codex. The agent has no network at run time.

In [ ]:
print((ROOT_TASK / "environment" / "Dockerfile").read_text())

### `tests/`

`tests/test.sh` is the verifier. It runs after the agent stops, and does these stages:

| Stage | Files | Output in `/logs/verifier/` |
|-------|-------|------------------------------|
| 0. Save the agent's change as a commit | | `agent.patch` |
| 1. Pre-written verifiers: restore the base test files (the agent cannot change a test so that it passes), add the hidden tests, and run all of them | `verify/` | `junit.xml`, `pytest.log` |
| 2. Agent-written tests: run them on the agent's fix and on the base | `lib/agent_tests.py` | `agent_tests.json` |
| 3. Validation agent and validation judge | `validate/validation_spec.toml`, `lib/validate.py` | `validation.json`, `validation_scripts/` |
| 4. Task rubric judge, taste judge, patch bloat | `judge/rubric.toml`, `judge/reference.patch`, `lib/judges.py` | `rubric.json`, `taste.json`, `bloat.json` |
| 5. Aggregate | `lib/aggregate.py` | `reward.json`, `reward_details.json` |

The LLM stages call Claude on Amazon Bedrock. The model of every stage is `VERIFIER_MODEL_ID` in `model_config.py` (Claude Sonnet 4.6). `judge/models.toml` has the default model, the temperature (0), and the turn budget of each stage, and the thresholds. The verifier records the models that ran in `reward_details.json`. In the verifier phase, Harbor changes the network of the container to an allowlist with only the Bedrock runtime endpoint, and gives it your AWS credentials. The agent phase never has them.

If an LLM stage fails (for example, throttling), or the validation judge discards the validation tests, the trial is **invalid**: there is no `reward.json`, and the trial is not counted. An infrastructure failure is not a failure of the agent.

The hidden tests use only the public API. They check behavior, not the design of the fix.

In [ ]:
print((HISTORY_TASK / "tests" / "verify" / "test_prior_tool_results_verify.py").read_text()[:3000])

### The validation spec

Each story has a description, a procedure, and cases. The validation agent gets the stories and the agent's change. It does not get the oracle. The driver runs each test once for each case, so the validation agent cannot hard-code the answers. The procedure tells it the one mock that it can use: a stub for the judge model.

In [ ]:
print((ROOT_TASK / "tests" / "validate" / "validation_spec.toml").read_text()[:3500])

### The rubric and the taste judge

The task rubric holds requirements that are important but that a test cannot check well, or that are preferences. A single failed criterion does not fail the task: the score must be above 0.5. The taste judge uses one global rubric for all tasks. It reads the conventions of the repository (for example `AGENTS.md` and `STYLE_GUIDE.md`) and compares the agent's change with the merged fix: 4 of 5 means "as good as the reference".

In [ ]:
print((ROOT_TASK / "tests" / "judge" / "rubric.toml").read_text())
print((ROOT_TASK / "tests" / "judge" / "models.toml").read_text())

### Leakage scan

Scan all files that the agent sees. Search for the PR number, the issue number, the merge commit, the name of the hidden test file, and lines that the fix adds. If the agent can see this data, the score measures search, not coding.

First, put a canary token in the environment. Make sure that the scan finds it. If the scan does not find it, the scan has a defect.

In [ ]:
def agent_visible_files(task_dir: Path):
    """Files that the agent sees."""
    yield task_dir / "instruction.md"
    yield from (p for p in (task_dir / "environment").rglob("*") if p.is_file())

def leakage_scan(task_dir: Path, needles: list[str]) -> list[tuple[str, str]]:
    hits = []
    for path in agent_visible_files(task_dir):
        text = path.read_text(errors="ignore").lower()
        for needle in needles:
            if needle and needle.lower() in text:
                hits.append((str(path.relative_to(task_dir)), needle))
    return hits

def needles_for(task_dir: Path) -> list[str]:
    """The PR and issue numbers, the merge commit, the hidden test file, and the long lines that the fix adds."""
    meta = read_toml(task_dir / "task.toml")["metadata"]
    origin = meta["origin"]
    needles = [f"#{n}" for n in origin["pr_numbers"] + origin["issue_numbers"]]
    needles += [f"pull/{n}" for n in origin["pr_numbers"]] + [origin["merge_commit"][:12]]
    needles += [Path(meta["verifier"]["hidden_tests"]).stem]
    patch = (task_dir / "solution" / "oracle.patch").read_text()
    needles += [line[1:].strip() for line in patch.splitlines()
                if line.startswith("+") and not line.startswith("+++") and len(line.strip()) >= 40]
    return needles

for task_dir in TASK_DIRS:
    # Canary test: the scan must find a known token.
    canary = "CANARY-7f3a91"
    canary_file = task_dir / "environment" / "canary.txt"
    canary_file.write_text(canary)
    try:
        assert leakage_scan(task_dir, [canary]), "The scan did not find the canary. The scan has a defect."
    finally:
        canary_file.unlink()

    needles = needles_for(task_dir)
    hits = leakage_scan(task_dir, needles)
    print(f"{task_dir.name}: {len(needles)} needles, " + ("no leaks." if not hits else f"leaks: {hits}"))

## 4. Select the Metrics

`tests/lib/aggregate.py` writes these signals to the reward file:

| Signal | Question |
|--------|----------|
| `reward`, `tasteful_solve` | Is the change correct, and would a maintainer merge it? 1 if every gate passes. |
| `correct` | Does the change work? 1 if the verifiers and the validation stories all pass. |
| `fail_to_pass`, `pass_to_pass`, `existing_suite` | What fraction of each group of pre-written tests passes? |
| `validation_score` | What fraction of the validation stories passes? |
| `rubric_score` | What fraction of the rubric criteria does the change fulfill? |
| `relative_taste`, `practice_alignment` | How good is the change compared with the merged fix, and how well does it fit the codebase (1-5)? |
| `patch_bloat` | How much larger is the change than the merged fix? |
| `agent_tests_added`, `agent_tests_reproduce_bug` | Did the agent add tests, and does one of them fail on the base? |

The resolved rate of an agent is the mean `correct`. The tasteful-solve rate is the mean `reward`. The difference between them tells you how often the agent writes a fix that works but that a reviewer would not merge.

The LLM signals have variance. A taste threshold of "more than 2 of 5" is low on purpose: it removes clearly bad changes, and a small change in the judge does not change the result.

## 5. Run the Tasks in Harbor

### Make sure that Docker runs

In [ ]:
ok = shutil.which("docker") and subprocess.run(["docker", "info"], capture_output=True).returncode == 0
assert ok, "Docker is not installed or does not run. Do the setup checks in notebook 01."
print("Docker runs.")

### Build the task images

Build each image before you run an agent. The build runs the full unit test suite on the base commit. If a test fails, the build fails. The first build downloads the base image, installs approximately 150 Python packages, and installs Codex. This can take some minutes.

The cell also builds the image of the egress proxy (see notebook 02, "Enforce the network allowlist"), so that two trials do not try to build it at the same time.

Each image is approximately 4 GB. If a build fails with `At least one invalid signature was encountered`, the Docker disk is full. Run `docker builder prune` to remove the build cache.

In [ ]:
# The egress proxy image, built one time before Harbor needs it.
r = subprocess.run(["docker", "build", "-q", "-t", "harvest-env-egress-proxy:1", "harvest_env/egress_proxy"],
                   capture_output=True, text=True, stdin=subprocess.DEVNULL)
print(f"{'OK ' if r.returncode == 0 else 'ERR'} egress proxy  {r.stdout.strip() or r.stderr.strip()[-300:]}")

for task_dir in TASK_DIRS:
    r = subprocess.run(["docker", "build", "-q", "-t", f"eval-{task_dir.name}", str(task_dir / "environment")],
                       capture_output=True, text=True, stdin=subprocess.DEVNULL)
    print(f"{'OK ' if r.returncode == 0 else 'ERR'} {task_dir.name}  {r.stdout.strip() or r.stderr.strip()[-300:]}")

### Check the git seal

In each image, `git log --all` must show one commit only, and `git remote` must show no remotes. `git status` must show no changes: the agent starts from the base commit. The hidden tests must not be in the image.

In [ ]:
SEAL_CHECK = ('echo "$(git log --all --oneline | wc -l) $(git remote | wc -l) $(git status --porcelain | wc -l) '
              '$(find / -name \'*_verify.py\' -not -path \'/proc/*\' 2>/dev/null | wc -l)"')

for task_dir in TASK_DIRS:
    out = subprocess.run(["docker", "run", "--rm", f"eval-{task_dir.name}", "sh", "-c", SEAL_CHECK],
                         capture_output=True, text=True, check=True).stdout
    commits, remotes, changed, hidden = map(int, out.split())
    print(f"{task_dir.name}: commits={commits} remotes={remotes} changed_files={changed} hidden_tests={hidden}")
    assert commits == 1, "More than one commit is reachable. The agent can find the fix."
    assert remotes == 0, "The repository has a remote."
    assert changed == 0, "The working tree is not the base commit."
    assert hidden == 0, "The hidden tests are in the image."
print("Seal checks passed.")

### Give AWS credentials to Harbor

Codex calls Amazon Bedrock in the agent phase. The validation agent and the judges call it in the verifier phase. This cell gets your credentials with boto3 and puts them in the environment of the kernel, as in notebook 02.

Temporary credentials expire, usually after one hour. The Harbor helper calls `refresh_credentials()` before each job, and each job of this notebook takes less than an hour. If a trial is invalid with `ExpiredToken` or `Signature expired`, run the cell again. Do not let your computer sleep during a run.

In [ ]:
import boto3

def refresh_credentials():
    """Put fresh AWS credentials from your profile or the default chain in the kernel environment."""
    # AWS_PROFILE comes from .env (see .env.example). If it is not set, boto3 uses the default credential chain.
    profile = os.environ.get("AWS_PROFILE") or None
    session = boto3.Session(profile_name=profile, region_name=AWS_REGION)
    creds = session.get_credentials()
    if creds is None:
        raise RuntimeError(
            f"boto3 found no AWS credentials. Profile used: {profile or 'none (default credential chain)'}.\n"
            f"Profiles that boto3 can see: {session.available_profiles}\n"
            "Set AWS_PROFILE in .env to one of these profiles. Then run the load_env cell and this cell again."
        )
    frozen = creds.get_frozen_credentials()
    os.environ["AWS_ACCESS_KEY_ID"] = frozen.access_key
    os.environ["AWS_SECRET_ACCESS_KEY"] = frozen.secret_key
    os.environ.pop("AWS_SESSION_TOKEN", None)
    if frozen.token:
        os.environ["AWS_SESSION_TOKEN"] = frozen.token
    os.environ["AWS_REGION"] = AWS_REGION
    return session, profile

session, profile = refresh_credentials()
arn = session.client("sts").get_caller_identity()["Arn"]
print(f"Profile: {profile or 'default credential chain'}")
print("AWS credentials found for:", arn.split(":")[-1])

### The Harbor helper

This helper runs Harbor with the proxy environment of notebook 02. It runs all tasks in one Harbor job, in `jobs/<job name>/`, writes the Harbor output to `jobs/<job name>.log`, and shows the end of the log. It gets fresh AWS credentials before each job. The job names of this notebook start with `coding-`, so they do not replace the results of notebook 02.

Harbor builds an image for each trial, and deletes it when the trial ends. The images of this notebook are approximately 4 GB each. The helper runs two trials at the same time (`N_CONCURRENT = 2`), so Docker needs approximately 8 GB of free disk space during a run. If the disk is full, set `N_CONCURRENT = 1`: the run takes twice as long.

In [ ]:
HARBOR_ENV = "harvest_env.docker_allowlist:ProxyAllowlistDockerEnvironment"
N_CONCURRENT = 2  # trials at the same time. Each trial builds an image of approximately 4 GB.

def harbor_run(job_name: str, *args: str, tasks: list[Path] | None = None) -> Path:
    """Run the tasks (default: all tasks in TASKS_DIR) as one Harbor job. Return the job directory."""
    job_dir = JOBS_DIR / job_name
    if job_dir.exists():
        shutil.rmtree(job_dir)  # a new run replaces the old results
    JOBS_DIR.mkdir(exist_ok=True)
    refresh_credentials()
    paths = [arg for task in tasks for arg in ("-p", str(task))] if tasks else ["-p", str(TASKS_DIR)]
    cmd = ["harbor", "run", *paths, "-e", HARBOR_ENV, "-o", str(JOBS_DIR), "--job-name", job_name,
           "-n", str(N_CONCURRENT), "-y", *args]
    env = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [os.getcwd(), os.environ.get("PYTHONPATH")]))}
    log = JOBS_DIR / f"{job_name}.log"
    with log.open("w") as f:
        r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, env=env)
    print("\n".join(log.read_text().splitlines()[-8:]))
    assert r.returncode == 0, f"harbor run failed. See {log}."
    return job_dir

def trial_rewards(job_dir: Path) -> dict[str, dict]:
    """Return the reward of each trial, by task directory name."""
    rewards = {}
    for result in job_dir.glob("*/result.json"):
        task = read_json(result)["task_name"].replace("/", "__")
        reward_file = result.parent / "verifier" / "reward.json"
        details_file = result.parent / "verifier" / "reward_details.json"
        if reward_file.exists():
            rewards[task] = read_json(reward_file)
        else:  # an invalid trial: reward_details.json tells why
            invalid = read_json(details_file)["invalid"] if details_file.exists() else ["no verifier output"]
            rewards[task] = {"reward": None, "invalid": invalid}
    return rewards

### Test the reward stack with the oracle

Before you spend tokens on an agent, run the `oracle` agent on one task. It applies the merged fix, so it must be a tasteful solve: all tests and validation stories pass, the rubric is above 0.5, and the taste scores are above 2. If it is not, a mechanism of the reward stack is broken.

This run uses no tokens for the agent. The validation agent and the judges use approximately 0.5 to 1 million input tokens. It takes approximately 4 minutes.

In [ ]:
SHOW = ["reward", "correct", "fail_to_pass", "pass_to_pass", "existing_suite", "validation_score",
        "rubric_score", "relative_taste", "practice_alignment", "patch_bloat"]

oracle = trial_rewards(harbor_run("coding-oracle", "-a", "oracle", tasks=[ROOT_TASK]))[ROOT_TASK.name]
print(ROOT_TASK.name)
print("  oracle:", {k: oracle.get(k) for k in SHOW})
assert oracle.get("reward") == 1.0, "The oracle is not a tasteful solve. See verifier/reward_details.json."
print("The oracle is a tasteful solve.")

### Calibration: what we ran before the workshop

When you build a task, test every mechanism on two known answers: the oracle (it must pass) and the `nop` agent, which does nothing (it must fail). The rules are:

- **Oracle:** a tasteful solve.
- **No-op agent:** `fail_to_pass`, `correct`, and `reward` are 0. `pass_to_pass` and `existing_suite` are 1. If an f2p test passes here, it does not test the bug. If a p2p test fails here, the test or the environment is wrong. The validation stories fail.

Senior SWE-Bench runs each one 3 times, and it rejects a task unless all 3 results are correct (pass^3). We ran each one time for both tasks:

| Task | Run | f2p | p2p | existing | validation | rubric | taste (rel. / practice) | bloat | reward |
|------|-----|-----|-----|----------|------------|--------|-------------------------|-------|--------|
| root agent (371) | oracle | 1.0 | 1.0 | 1.0 | 2/2 stories | 1.0 | 4.0 / 4.6 | 1.0 | **1** |
| root agent (371) | no-op | 0.0 | 1.0 | 1.0 | 0/2 | 0.5 | – | – | 0 |
| tool history (338) | oracle | 1.0 | 1.0 | 1.0 | 2/2 stories | 1.0 | 4.0 / 4.6 | 1.0 | **1** |
| tool history (338) | no-op | 0.0 | 1.0 | 1.0 | 0/2 | 0.33 | – | – | 0 |

The no-op agent gets some rubric credit, because an empty change fulfills the pass-to-pass criteria ("no interface change"). This is why the rubric does not gate a solve alone.

To run the full calibration yourself (all tasks, oracle and no-op agent, approximately 15 minutes), set `RUN_CALIBRATION = True`.

In [ ]:
RUN_CALIBRATION = False  # approximately 15 minutes and 2 to 4 million judge tokens

if RUN_CALIBRATION:
    oracle_rewards = trial_rewards(harbor_run("coding-oracle-all", "-a", "oracle"))
    nop_rewards = trial_rewards(harbor_run("coding-nop", "-a", "nop"))
    for task_dir in TASK_DIRS:
        oracle, nop = oracle_rewards[task_dir.name], nop_rewards[task_dir.name]
        print(task_dir.name)
        print("  oracle:", {k: oracle.get(k) for k in SHOW})
        print("  nop:   ", {k: nop.get(k) for k in SHOW})
        assert oracle.get("reward") == 1.0, "The oracle is not a tasteful solve."
        assert nop.get("correct") == 0.0 and nop.get("fail_to_pass") == 0.0, "The no-op agent passes a correctness check."
        assert nop.get("pass_to_pass") == 1.0 and nop.get("existing_suite") == 1.0, "A test fails on the base."
        assert nop.get("validation_score", 1) < 1.0, "The validation stories pass without a fix."
    print("The oracle is a tasteful solve, and the no-op agent fails.")
else:
    print("Skipped. Set RUN_CALIBRATION = True to run it.")

### Run Codex

Now Codex gets each issue report, and writes a fix. The flags are the same as in notebook 02:

| Flag | Purpose |
|------|---------|
| `-a codex -m CODEX_MODEL_ID` | Run Harbor's Codex agent with the model on Bedrock (from `model_config.py`) |
| `--ak config=codex-bedrock.toml` | Give Codex the Bedrock provider settings |
| `--ak disable_web_search=true` | Turn off web search. The allowlist also blocks it. |
| `--ae AWS_...=${AWS_...}` | Give the AWS credentials to Codex |

### Number of trials

`N_ATTEMPTS` sets the number of trials for each task. The default is 1 (pass@1): one trial for each task, two trials in total.

> **Warning:** A coding trial runs longer than a review trial, because the agent runs the tests many times, and then the validation agent and the judges run. With the defaults, this cell takes approximately 10 minutes. More trials use much CPU, memory, and disk on a laptop, and more tokens. For more trials or more tasks, run Harbor on external compute, for example Amazon EC2 or Lambda MicroVMs.

In [ ]:
N_ATTEMPTS = 1  # trials for each task. Read the warning above before you increase this value.

codex_job = harbor_run(
    "coding-codex",
    "-k", str(N_ATTEMPTS),
    "-a", "codex", "-m", CODEX_MODEL_ID,
    "--ak", "config=codex-bedrock.toml",
    "--ak", "disable_web_search=true",
    "--ae", "AWS_ACCESS_KEY_ID=${AWS_ACCESS_KEY_ID}",
    "--ae", "AWS_SECRET_ACCESS_KEY=${AWS_SECRET_ACCESS_KEY}",
    "--ae", "AWS_SESSION_TOKEN=${AWS_SESSION_TOKEN:-}",  # empty for long-term access keys
    "--ae", f"AWS_REGION={AWS_REGION}",
)

## 6. Calculate the Results

For each trial, this cell reads `reward_details.json` and shows each gate of the tasteful solve, the tests that failed, the validation stories, and the files that the agent changed.

In [ ]:
def load_rows(job_dir: Path) -> list[dict]:
    """Return one row for each trial."""
    rows = []
    for result_path in sorted(job_dir.glob("*/result.json")):
        trial_dir = result_path.parent
        result = read_json(result_path)
        details_file = trial_dir / "verifier" / "reward_details.json"
        details = read_json(details_file) if details_file.exists() else {"invalid": ["no verifier output"], "flat": {}}
        rows.append({
            "task": result["task_name"].replace("/", "__"),
            "trial": trial_dir.name,
            "invalid": details["invalid"],
            "checks": details.get("checks", {}),
            **details.get("flat", {}),
            "failed_tests": details.get("failed_tests", []),
            "stories": {k: f"{v['passed']}/{v['total']}" for k, v in details.get("validation", {}).get("stories", {}).items()},
            "taste": {k: v["score"] for k, v in details.get("taste", {}).get("dimensions", {}).items()},
            "input_tokens": (result.get("agent_result") or {}).get("n_input_tokens"),
            "error": (result.get("exception_info") or {}).get("exception_type"),
        })
    return rows

rows = load_rows(codex_job)
for r in rows:
    if r["error"] or r["invalid"]:
        print(f"INVALID {r['task']}: {r['error'] or '; '.join(r['invalid'])}")
        continue
    status = "TASTEFUL SOLVE" if r["reward"] == 1.0 else ("CORRECT, NOT TASTEFUL" if r["correct"] == 1.0 else "FAIL")
    print(f"{status}  {r['task']}  (agent input tokens: {r['input_tokens']})")
    for gate, ok in r["checks"].items():
        print(f"    {'pass' if ok else 'FAIL'}  {gate}")
    print(f"    tests:      f2p={r['fail_to_pass']} p2p={r['pass_to_pass']} existing={r['existing_suite']}")
    print(f"    validation: {r['stories']}")
    print(f"    rubric={r['rubric_score']} relative_taste={r['relative_taste']} practice={r['practice_alignment']} bloat={r['patch_bloat']}")
    print(f"    agent tests: added={r['agent_tests_added']:.0f} reproduce_bug={r['agent_tests_reproduce_bug']:.0f}")
    for line in r["failed_tests"][:10]:
        print(f"    failed: {line}")

scored = [r for r in rows if not (r["error"] or r["invalid"])]
if scored:
    print(f"\nCorrect:         {sum(r['correct'] for r in scored):.0f}/{len(scored)} trials")
    print(f"Tasteful solves: {sum(r['reward'] for r in scored):.0f}/{len(scored)} trials")

### Examine each failure

The score tells you that the agent failed. The trajectory and the patch tell you why. Each trial has:

- `jobs/coding-codex/<trial>/agent/`: the trajectory, each command and message of the agent.
- `jobs/coding-codex/<trial>/verifier/agent.patch`: the changes of the agent.
- `jobs/coding-codex/<trial>/verifier/failed_tests.txt` and `pytest.log`: the tests that failed, and why.
- `jobs/coding-codex/<trial>/verifier/validation_scripts/` and `validation.json`: the tests that the validation agent wrote, their results, and the review of the validation judge.
- `jobs/coding-codex/<trial>/verifier/rubric.json` and `taste.json`: the score and the reason for each criterion and each taste dimension.

**The results change from run to run.** With two tasks and one trial for each, the resolved rate can be only 0, 0.5, or 1. To get a more stable result, increase `N_ATTEMPTS`.

**Use a coding agent to read the trajectories.** For example:

```text
Read the Harbor job in jobs/coding-codex/. For each trial with reward 0, compare
verifier/agent.patch with the task's solution/oracle.patch, read verifier/failed_tests.txt,
and read the trajectory in agent/. Tell me what the agent thought the cause was, which tests it
ran, and why its fix fails the hidden tests. For a trial that is correct but not tasteful, read
verifier/taste.json and verifier/rubric.json and tell me if you agree with the low scores.
Do not change files.
```

**Find the cause.** Each failure has one of these causes:

| Cause | Example | What to do |
|-------|---------|------------|
| **The agent fixed the wrong thing** | The agent changed the evaluator prompt, not the extractor. | Use the eval to improve the agent: its prompt, its tools, or its procedure (for example, "write a failing test first"). |
| **The fix is not complete** | `fail_to_pass` is 0.8: the fix fails on spans that are not in time order. | Same as above. Also make sure that the instruction asks for this behavior. |
| **The fix breaks something** | `existing_suite` is less than 1. | Same as above. The agent did not run the full test suite. |
| **The hidden test checks the design, not the behavior** | A correct fix fails because a test expects one place in the history. | Change the test so that a different correct fix passes. Then run the oracle and the no-op agent again. |
| **The instruction does not ask for the behavior** | A test checks an edge case that the issue report does not tell. | Add the behavior to the instruction, or remove the test. |
| **The fix works, but it is not tasteful** | `correct` is 1, but `relative_taste` is 2: the agent patched the symptom in the evaluator, not the cause in the extractor. | Use the eval to improve the agent. Read `taste.json` first: if you do not agree with the judge, change the judge prompt, and run the oracle again. |
| **The validation tests are wrong** | The validation judge discarded the trial, or a story fails with an import error in the test code. | Make the procedure in `validation_spec.toml` more precise. Then run the oracle and the no-op agent again. |
| **The environment is broken** | A command failed because a tool is missing. | Repair the Dockerfile, then build the images again. |

## 7. Look at the Results in Harbor View

`harbor view` starts a web server that shows each job, each trial, the reward, and the full trajectory of the agent. It shows the jobs of notebooks 02 and 03.

This cell starts the server in the background, and shows the URL. Open the URL in your browser.

In [ ]:
import time

viewer_log = JOBS_DIR / "harbor-view.log"
viewer = subprocess.Popen(["harbor", "view", str(JOBS_DIR), "--port", "8080-8089"],
                          stdout=viewer_log.open("w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)

# Wait until the server shows its URL.
urls = []
for _ in range(120):
    urls = re.findall(r"https?://[\w.:-]+", viewer_log.read_text())
    if urls or viewer.poll() is not None:
        break
    time.sleep(1)

if urls:
    print(f"Harbor view runs at {urls[0]}")
else:
    print(viewer_log.read_text()[-2000:])
    raise RuntimeError(f"harbor view did not start. See {viewer_log}.")

When you are done, run this cell to stop the server.

In [ ]:
viewer.terminate()
print("Harbor view stopped.")